In [3]:

import os
os.getcwd()

'c:\\dev\\accountant_agent_v3\\laboratoire'

In [5]:
import sys
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY") 
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
MODELS="openai/gpt-oss-120b"
MODEL = "openai:gpt-4o-mini"

llm_groq = init_chat_model(model=MODEL, temperature=0)

c:\dev\accountant_agent_v3\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## EVALUATION

In [ ]:
#!/usr/bin/env python3
# ============================================================================
# MOROCCAN ACCOUNTING RAG EVALUATION - REALISTIC VERSION
# With simulated model outputs and RAGAS metrics
# ============================================================================

import json
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass
import re
import sys
from difflib import SequenceMatcher
import subprocess

# Install RAGAS if not present
try:
    from ragas import evaluate
    from ragas.metrics.collections import (
        faithfulness,
        answer_relevancy,
        context_recall,
        context_precision,

    )
    HAS_RAGAS = True
except ImportError:
    HAS_RAGAS = False
    print("⚠️  RAGAS not available - using custom metrics only")

# ============================================================================
# LOAD THE EVALUATION DATA
# ============================================================================

with open("data.json") as f:
    evaluation_data = json.load(f)
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def extract_amounts(text: str) -> List[float]:
    """Extract all numeric amounts from text"""
    pattern = r'(\d+(?:[.,]\d+)?)\s*(?:MAD|USD)?'
    matches = re.findall(pattern, text)
    return [float(m.replace(',', '.')) for m in matches]

def calculate_similarity(expected: str, actual: str) -> float:
    """Calculate string similarity ratio"""
    return SequenceMatcher(None, expected.lower(), actual.lower()).ratio()

def extract_account_codes(text: str) -> List[str]:
    """Extract account codes"""
    pattern = r'\b(\d{4,5})\b'
    return re.findall(pattern, text)

def check_debit_credit_match(expected_debits: float, expected_credits: float, 
                             actual_text: str, expected_debits_val: float = None,
                             expected_credits_val: float = None) -> Tuple[bool, float]:
    """Check if debits match credits"""
    actual_amounts = extract_amounts(actual_text)
    
    if not actual_amounts or len(actual_amounts) < 2:
        return False, 0.0
    
    # Sum all amounts (simplified check)
    actual_sum = sum(actual_amounts)
    expected_sum = expected_debits_val + expected_credits_val if expected_debits_val and expected_credits_val else 0
    
    if expected_sum == 0:
        return False, 0.0
    
    match_ratio = min(actual_sum, expected_sum) / max(actual_sum, expected_sum)
    return match_ratio >= 0.99, match_ratio

def evaluate_account_codes(expected: List[str], actual_text: str) -> float:
    """Evaluate account code correctness"""
    extracted = set(extract_account_codes(actual_text))
    expected_set = set(expected)
    
    if not expected_set:
        return 0.5
    
    matches = len(extracted & expected_set)
    return matches / len(expected_set)

def evaluate_tva_handling(expected_text: str, actual_text: str) -> float:
    """Evaluate TVA calculation and handling"""
    expected_has_tva = '34552' in expected_text or '34551' in expected_text or 'TVA' in expected_text
    expected_has_no_tva = 'Pas de TVA' in expected_text or 'taxe spécifique' in expected_text
    
    actual_has_tva = '34552' in actual_text or '34551' in actual_text or 'TVA' in actual_text
    
    if expected_has_no_tva:
        # Should NOT have TVA
        return 1.0 if not actual_has_tva else 0.5
    elif expected_has_tva:
        # Should have TVA
        return 1.0 if actual_has_tva else 0.4
    
    return 0.5

def calculate_bleu_score(expected: str, actual: str) -> float:
    """Simple BLEU-like score (word overlap)"""
    expected_words = set(expected.lower().split())
    actual_words = set(actual.lower().split())
    
    if not expected_words:
        return 0.0
    
    overlap = len(expected_words & actual_words)
    return overlap / len(expected_words)

# ============================================================================
# MAIN EVALUATION
# ============================================================================

def evaluate_rag_system(eval_data: List[Dict]) -> pd.DataFrame:
    """Comprehensive evaluation comparing expected vs actual outputs"""
    results = []
    
    for case in eval_data:
        case_id = case['case_id']
        expected_entry = case['expected_entry']
        model_output = case['model_output']
        context_retrieved = case['context_retrieved']
        expected_accounts = case['expected_accounts']
        expected_debits = case['expected_debits']
        expected_credits = case['expected_credits']
        
        # 1. Account Code Accuracy
        account_accuracy = evaluate_account_codes(expected_accounts, model_output)
        
        # 2. Debit/Credit Balance
        is_balanced, balance_ratio = check_debit_credit_match(
            expected_debits, expected_credits, model_output,
            expected_debits, expected_credits
        )
        
        # 3. TVA Handling
        tva_accuracy = evaluate_tva_handling(expected_entry, model_output)
        
        # 4. Context Relevance (similarity between context and expected entry)
        context_relevance = calculate_similarity(expected_entry, context_retrieved)
        
        # 5. Output Completeness (similarity between expected and actual)
        output_similarity = calculate_similarity(expected_entry, model_output)
        
        # 6. BLEU-like metric (word overlap)
        bleu_score = calculate_bleu_score(expected_entry, model_output)
        
        # Calculate weighted overall score
        overall_score = (
            account_accuracy * 0.25 +
            balance_ratio * 0.25 +
            tva_accuracy * 0.20 +
            context_relevance * 0.10 +
            output_similarity * 0.15 +
            bleu_score * 0.05
        )
        
        results.append({
            'Case': case_id,
            'Query': case['user_input'][:50] + '...',
            'Account Accuracy': account_accuracy,
            'Balance Match': balance_ratio,
            'TVA Handling': tva_accuracy,
            'Context Quality': context_relevance,
            'Output Match': output_similarity,
            'BLEU Score': bleu_score,
            'Overall': overall_score,
            'Status': '✓' if overall_score >= 0.8 else '△' if overall_score >= 0.6 else '✗'
        })
    
    return pd.DataFrame(results)

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    print('\n' + '='*100)
    print('MOROCCAN ACCOUNTING RAG SYSTEM - REALISTIC EVALUATION')
    print('Comparing Model Outputs vs Expected Results')
    print('='*100 + '\n')
    
    print(f'📊 Total Test Cases: {len(evaluation_data)}\n')
    
    # Run evaluation
    results_df = evaluate_rag_system(evaluation_data)
    
    # Display detailed results
    print('DETAILED RESULTS (Model Output vs Expected):')
    print('-'*100)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 50)
    pd.set_option('display.float_format', '{:.3f}'.format)
    print(results_df.to_string(index=False))
    print()
    
    # ========================================================================
    # AGGREGATE METRICS
    # ========================================================================
    
    print('\n' + '='*100)
    print('AGGREGATE METRICS & COMPONENT BREAKDOWN')
    print('='*100 + '\n')
    
    metrics = {
        'Account Code Accuracy': results_df['Account Accuracy'].mean(),
        'Debit/Credit Balance': results_df['Balance Match'].mean(),
        'TVA Handling': results_df['TVA Handling'].mean(),
        'Context Quality': results_df['Context Quality'].mean(),
        'Output Match (Similarity)': results_df['Output Match'].mean(),
        'BLEU Score (Word Overlap)': results_df['BLEU Score'].mean(),
        'Overall RAG Score': results_df['Overall'].mean()
    }
    
    print('Component Performance:\n')
    for metric, value in metrics.items():
        bar_length = int(value * 50)
        bar = '█' * bar_length + '░' * (50 - bar_length)
        status = '✓ Good' if value >= 0.80 else '△ Fair' if value >= 0.60 else '✗ Poor'
        print(f'{metric:<35} {bar} {value:>7.2%}  {status}')
    
    # ========================================================================
    # DIAGNOSTIC ANALYSIS
    # ========================================================================
    
    print('\n' + '='*100)
    print('DIAGNOSTIC ANALYSIS')
    print('='*100 + '\n')
    
    perfect = (results_df['Overall'] >= 0.95).sum()
    excellent = (results_df['Overall'] >= 0.85).sum()
    good = (results_df['Overall'] >= 0.70).sum()
    acceptable = (results_df['Overall'] >= 0.60).sum()
    poor = (results_df['Overall'] < 0.60).sum()
    
    print(f'Perfect (>= 95%):        {perfect:2d}/{len(results_df)} cases')
    print(f'Excellent (85-95%):      {excellent - perfect:2d}/{len(results_df)} cases')
    print(f'Good (70-85%):           {good - excellent:2d}/{len(results_df)} cases')
    print(f'Acceptable (60-70%):     {acceptable - good:2d}/{len(results_df)} cases')
    print(f'Poor (< 60%):            {poor:2d}/{len(results_df)} cases')
    print()
    
    # Problem areas
    weakest = min(metrics.items(), key=lambda x: x[1])
    strongest = max(metrics.items(), key=lambda x: x[1])
    
    print(f'🏆 Strongest: {strongest[0]}: {strongest[1]:.2%}')
    print(f'⚠️  Weakest:   {weakest[0]}: {weakest[1]:.2%}')
    print()
    
    # Worst performing cases
    worst_cases = results_df.nsmallest(3, 'Overall')
    print('Worst Performing Cases:')
    for _, row in worst_cases.iterrows():
        print(f'  Case {int(row["Case"])}: {row["Overall"]:.2%} - {row["Status"]}')
    
    # ========================================================================
    # SUMMARY
    # ========================================================================
    
    print('\n' + '='*100)
    print('SUMMARY')
    print('='*100 + '\n')
    
    overall = metrics['Overall RAG Score']
    
    if overall >= 0.90:
        status = '✓ EXCELLENT - Production ready'
        icon = '🟢'
    elif overall >= 0.80:
        status = '✓ GOOD - Minor issues'
        icon = '🟢'
    elif overall >= 0.70:
        status = '△ ACCEPTABLE - Needs improvements'
        icon = '🟡'
    elif overall >= 0.60:
        status = '✗ POOR - Significant work needed'
        icon = '🔴'
    else:
        status = '✗ CRITICAL - Major redesign'
        icon = '🔴'
    
    print(f'Overall RAG Score: {overall:.2%}')
    print(f'Status: {icon} {status}\n')
    
    # Recommendations
    print('KEY ISSUES TO ADDRESS:')
    print('-'*100)
    
    if metrics['Account Code Accuracy'] < 0.85:
        print('1. Account Code Selection')
        print('   → Improve retrieval of PCGM account codes from documents')
        print('   → Implement fuzzy matching for similar account names\n')
    
    if metrics['Debit/Credit Balance'] < 0.85:
        print('2. Debit/Credit Balance Validation')
        print('   → Add post-processing validation for entry balance')
        print('   → Include balance check in LLM prompt\n')
    
    if metrics['TVA Handling'] < 0.85:
        print('3. TVA Calculation & Rules')
        print('   → Create comprehensive TVA lookup (20%, 14%, 7%, 0%, exempt)')
        print('   → Handle special cases: assurances, carburant, foreign invoices\n')
    
    if metrics['Output Match (Similarity)'] < 0.75:
        print('4. Output Format Consistency')
        print('   → Standardize journal entry formatting in prompts')
        print('   → Add few-shot examples to LLM prompt\n')
    
    # ========================================================================
    # EXPORT RESULTS
    # ========================================================================
    
    print('='*100)
    print('EXPORT RESULTS')
    print('='*100 + '\n')
    
    # CSV export
    csv_file = 'evaluation/rag_evaluation_realistic.csv'
    results_df.to_csv(csv_file, index=False)
    print(f'✓ Detailed results: {csv_file}')
    
    # Metrics summary
    metrics_df = pd.DataFrame([metrics])
    metrics_file = 'evaluation/rag_metrics_summary.csv'
    metrics_df.to_csv(metrics_file, index=False)
    print(f'✓ Metrics summary: {metrics_file}')
    
    # JSON export
    json_results = {
        'evaluation': 'Moroccan Accounting RAG System',
        'total_cases': len(evaluation_data),
        'overall_score': float(overall),
        'status': status.replace('✓', '').replace('✗', '').replace('△', '').strip(),
        'metrics': {k: float(v) for k, v in metrics.items()},
        'case_breakdown': results_df[['Case', 'Overall', 'Status']].to_dict('records')
    }
    
    json_file = 'evaluation/rag_evaluation_realistic.json'
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(json_results, f, ensure_ascii=False, indent=2)
    print(f'✓ JSON export: {json_file}')
    
    print('\n' + '='*100)
    print('✓ Evaluation Complete!')
    print('='*100 + '\n')

if __name__ == '__main__':
    main()

⚠️  RAGAS not available - using custom metrics only

MOROCCAN ACCOUNTING RAG SYSTEM - REALISTIC EVALUATION
Comparing Model Outputs vs Expected Results

📊 Total Test Cases: 15

DETAILED RESULTS (Model Output vs Expected):
----------------------------------------------------------------------------------------------------
 Case                                                 Query  Account Accuracy  Balance Match  TVA Handling  Context Quality  Output Match  BLEU Score  Overall Status
    1 Facture Maroc Telecom: 300 MAD TTC. Internet Fibre...             0.667          0.006         1.000            0.579         0.865       0.684    0.590      ✗
    2 Achat d'un PC portable HP Victus pour 8500 MAD che...             0.333          0.694         0.400            0.225         0.590       0.421    0.469      ✗
    3 Paiement loyer bureau: 4000 MAD par chèque. Query:...             1.000          0.107         0.500            0.530         0.910       0.727    0.603      △
    4 Facture 